## Calculating AQI (Air Quality Index) in India with SNN approch

This notebook provides a workflow for calculating Air Quality Index. The formula is as per CPCB's official AQI Calculator: https://app.cpcbccr.com/ccr_docs/How_AQI_Calculated.pdf

I added SNN functionality, but most of the AQI caluclation remains the same

## Preparing data
The dataset used is hourly air quality data (2015 - 2020) from various measuring stations across India: https://www.kaggle.com/rohanrao/air-quality-data-in-india

We'll use one city (Thiruvananthapuram in Kerala) that has two stations and compare it with the actual AQI values present in the data at station, city, hour and day level to confirm the calculations are correct.


In [20]:
## importing packages
import numpy as np
import pandas as pd


In [21]:
## defining constants
PATH_STATION_HOUR = "./data/station_hour.csv"
PATH_STATIONS = "./data/stations.csv"

In [22]:
## importing data and subsetting the station
df = pd.read_csv(PATH_STATION_HOUR, parse_dates = ["Datetime"])
stations = pd.read_csv(PATH_STATIONS)

df = df.merge(stations, on = "StationId")

df.sort_values(["StationId", "Datetime"], inplace = True)
df.reset_index(drop = True, inplace = True)
print(f"Loaded {len(df):,} rows, {df.StationId.nunique()} stations")


/tmp/ipykernel_33109/3461417722.py:2: DtypeWarning: Columns (0: AQI_Bucket) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PATH_STATION_HOUR, parse_dates = ["Datetime"])


Loaded 2,589,083 rows, 110 stations


## Formula
![](https://i.imgur.com/vQR5Zy0.png)

* The AQI calculation uses 7 measures: **PM2.5, PM10, SO2, NOx, NH3, CO and O3**.
* For **PM2.5, PM10, SO2, NOx and NH3** the average value in last 24-hrs is used with the condition of having at least 16 values.
* For **CO and O3** the maximum value in last 8-hrs is used.
* Each measure is converted into a Sub-Index based on pre-defined groups.
* Sometimes measures are not available due to lack of measuring or lack of required data points.
* Final AQI is the maximum Sub-Index with the condition that at least one of PM2.5 and PM10 should be available and at least three out of the seven should be available.


In [23]:
df["PM10_24hr_avg"] = df.groupby("StationId")["PM10"].rolling(window = 24, min_periods = 16).mean().values
df["PM2.5_24hr_avg"] = df.groupby("StationId")["PM2.5"].rolling(window = 24, min_periods = 16).mean().values
df["SO2_24hr_avg"] = df.groupby("StationId")["SO2"].rolling(window = 24, min_periods = 16).mean().values
df["NOx_24hr_avg"] = df.groupby("StationId")["NOx"].rolling(window = 24, min_periods = 16).mean().values
df["NH3_24hr_avg"] = df.groupby("StationId")["NH3"].rolling(window = 24, min_periods = 16).mean().values
df["CO_8hr_max"] = df.groupby("StationId")["CO"].rolling(window = 8, min_periods = 1).max().values
df["O3_8hr_max"] = df.groupby("StationId")["O3"].rolling(window = 8, min_periods = 1).max().values


## PM2.5 (Particulate Matter 2.5-micrometer)
PM2.5 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:


In [24]:
## PM2.5 Sub-Index calculation
def get_PM25_subindex(x):
    if x <= 30:
        return x * 50 / 30
    elif x <= 60:
        return 50 + (x - 30) * 50 / 30
    elif x <= 90:
        return 100 + (x - 60) * 100 / 30
    elif x <= 120:
        return 200 + (x - 90) * 100 / 30
    elif x <= 250:
        return 300 + (x - 120) * 100 / 130
    elif x > 250:
        return 400 + (x - 250) * 100 / 130
    else:
        return 0

df["PM2.5_SubIndex"] = df["PM2.5_24hr_avg"].apply(lambda x: get_PM25_subindex(x))


## PM10 (Particulate Matter 10-micrometer)
PM10 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:


In [25]:
## PM10 Sub-Index calculation
def get_PM10_subindex(x):
    if x <= 50:
        return x
    elif x <= 100:
        return x
    elif x <= 250:
        return 100 + (x - 100) * 100 / 150
    elif x <= 350:
        return 200 + (x - 250)
    elif x <= 430:
        return 300 + (x - 350) * 100 / 80
    elif x > 430:
        return 400 + (x - 430) * 100 / 80
    else:
        return 0

df["PM10_SubIndex"] = df["PM10_24hr_avg"].apply(lambda x: get_PM10_subindex(x))


## SO2 (Sulphur Dioxide)
SO2 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:

In [26]:
## SO2 Sub-Index calculation
def get_SO2_subindex(x):
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 380:
        return 100 + (x - 80) * 100 / 300
    elif x <= 800:
        return 200 + (x - 380) * 100 / 420
    elif x <= 1600:
        return 300 + (x - 800) * 100 / 800
    elif x > 1600:
        return 400 + (x - 1600) * 100 / 800
    else:
        return 0

df["SO2_SubIndex"] = df["SO2_24hr_avg"].apply(lambda x: get_SO2_subindex(x))


## NOx (Any Nitric x-oxide)
NOx is measured in ppb (parts per billion). The predefined groups are defined in the function below:


In [27]:
## NOx Sub-Index calculation
def get_NOx_subindex(x):
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 180:
        return 100 + (x - 80) * 100 / 100
    elif x <= 280:
        return 200 + (x - 180) * 100 / 100
    elif x <= 400:
        return 300 + (x - 280) * 100 / 120
    elif x > 400:
        return 400 + (x - 400) * 100 / 120
    else:
        return 0

df["NOx_SubIndex"] = df["NOx_24hr_avg"].apply(lambda x: get_NOx_subindex(x))


## NH3 (Ammonia)
NH3 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:

In [28]:
## NH3 Sub-Index calculation
def get_NH3_subindex(x):
    if x <= 200:
        return x * 50 / 200
    elif x <= 400:
        return 50 + (x - 200) * 50 / 200
    elif x <= 800:
        return 100 + (x - 400) * 100 / 400
    elif x <= 1200:
        return 200 + (x - 800) * 100 / 400
    elif x <= 1800:
        return 300 + (x - 1200) * 100 / 600
    elif x > 1800:
        return 400 + (x - 1800) * 100 / 600
    else:
        return 0

df["NH3_SubIndex"] = df["NH3_24hr_avg"].apply(lambda x: get_NH3_subindex(x))


## CO (Carbon Monoxide)
CO is measured in mg / m3 (milligrams per cubic meter of air). The predefined groups are defined in the function below:

In [29]:
## CO Sub-Index calculation
def get_CO_subindex(x):
    if x <= 1:
        return x * 50 / 1
    elif x <= 2:
        return 50 + (x - 1) * 50 / 1
    elif x <= 10:
        return 100 + (x - 2) * 100 / 8
    elif x <= 17:
        return 200 + (x - 10) * 100 / 7
    elif x <= 34:
        return 300 + (x - 17) * 100 / 17
    elif x > 34:
        return 400 + (x - 34) * 100 / 17
    else:
        return 0

df["CO_SubIndex"] = df["CO_8hr_max"].apply(lambda x: get_CO_subindex(x))


## O3 (Ozone or Trioxygen)
O3 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:

In [30]:
## O3 Sub-Index calculation
def get_O3_subindex(x):
    if x <= 50:
        return x * 50 / 50
    elif x <= 100:
        return 50 + (x - 50) * 50 / 50
    elif x <= 168:
        return 100 + (x - 100) * 100 / 68
    elif x <= 208:
        return 200 + (x - 168) * 100 / 40
    elif x <= 748:
        return 300 + (x - 208) * 100 / 539
    elif x > 748:
        return 400 + (x - 400) * 100 / 539
    else:
        return 0

df["O3_SubIndex"] = df["O3_8hr_max"].apply(lambda x: get_O3_subindex(x))


## AQI
The final AQI is the maximum Sub-Index among the available sub-indices with the condition that at least one of PM2.5 and PM10 should be available and at least three out of the seven should be available.

There is no theoretical upper value of AQI but its rare to find values over 1000.

The pre-defined buckets of AQI are as follows:
![](https://i.imgur.com/XmnE0rT.png)


In [ ]:
## AQI bucketing
def get_AQI_bucket(x):
    if pd.isna(x):
        return np.NaN
    if x <= 50:
        return "Good"
    elif x <= 100:
        return "Satisfactory"
    elif x <= 200:
        return "Moderate"
    elif x <= 300:
        return "Poor"
    elif x <= 400:
        return "Very Poor"
    elif x > 400:
        return "Severe"
    else:
        return np.NaN

df["Checks"] = (df["PM2.5_SubIndex"] > 0).astype(int) + \
                (df["PM10_SubIndex"] > 0).astype(int) + \
                (df["SO2_SubIndex"] > 0).astype(int) + \
                (df["NOx_SubIndex"] > 0).astype(int) + \
                (df["NH3_SubIndex"] > 0).astype(int) + \
                (df["CO_SubIndex"] > 0).astype(int) + \
                (df["O3_SubIndex"] > 0).astype(int)

df["AQI_calculated"] = round(df[["PM2.5_SubIndex", "PM10_SubIndex", "SO2_SubIndex", "NOx_SubIndex",
                                 "NH3_SubIndex", "CO_SubIndex", "O3_SubIndex"]].max(axis = 1))
df.loc[df["PM2.5_SubIndex"] + df["PM10_SubIndex"] <= 0, "AQI_calculated"] = np.NaN
df.loc[df.Checks < 3, "AQI_calculated"] = np.NaN

df["AQI_bucket_calculated"] = df["AQI_calculated"].apply(lambda x: get_AQI_bucket(x))
df[~df.AQI_calculated.isna()].head(13)


AttributeError: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.

In [ ]:
df[~df.AQI_calculated.isna()].AQI_bucket_calculated.value_counts()

Satisfactory    17568
Good             5089
Moderate         4487
Poor              121
Name: AQI_bucket_calculated, dtype: int64

Matches perfectly. In case of any discrepancy or bug or issue, feel free to comment here or share on the [dataset page](https://www.kaggle.com/rohanrao/air-quality-data-in-india).

# SNN Part

I kept most of the AQI calcualtion as in the tutorial (except using all stations and local data paths)

Below we turn the 7 raw pollutant readings into features, then build a classifier using spikes. The main chnages are:

 - **Input encoding** floats with `spikegen.rate` to binary spike trains
 - Replacing `ReLU` with `snn.Leaky` (LIF neurons) as **activation function**
 - **Forward pass** is nowa  loop over timesteps that accumulates output spikes instead of default single pass

In [ ]:
## SNN imports (the only new dependencies)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

import snntorch as snn
from snntorch import spikegen

### Feature preparation
7 raw pollutant columns as features and the calculated AQI bucket as the target label. Rows with missing AQI are dropped. Features are Z-score normalised then clipped to [0, 1] so they can be treated as spike probabilities

In [ ]:
FEATURES = ["PM2.5", "PM10", "SO2", "NOx", "NH3", "CO", "O3"]
BUCKET_TO_INT = {"Good": 0, "Satisfactory": 1, "Moderate": 2, "Poor": 3, "Very Poor": 4, "Severe": 5}
INT_TO_BUCKET = {v: k for k, v in BUCKET_TO_INT.items()}
NUM_STEPS = 25

snn_df = df[FEATURES + ["AQI_bucket_calculated"]].copy()
snn_df[FEATURES] = snn_df[FEATURES].ffill()
snn_df.dropna(subset=["AQI_bucket_calculated"], inplace=True)
snn_df = snn_df[snn_df["AQI_bucket_calculated"] != "0"]

snn_df["label"] = snn_df["AQI_bucket_calculated"].map(BUCKET_TO_INT)
snn_df.dropna(subset=["label"], inplace=True)
snn_df["label"] = snn_df["label"].astype(int)

print(f"Samples with valid AQI bucket: {len(snn_df):,}")
print(snn_df["AQI_bucket_calculated"].value_counts())

## Temporal split 80% train and 20% testing, normalization + loading data

In [ ]:
## Temporal train/test split (80/20) and normalisation
split = int(len(snn_df) * 0.8)
train_df = snn_df.iloc[:split]
test_df  = snn_df.iloc[split:]

mean = train_df[FEATURES].mean()
std  = train_df[FEATURES].std().replace(0, 1)

def normalize(part):
    X = ((part[FEATURES] - mean) / std).clip(0, 1).values.astype(np.float32)
    y = part["label"].values
    return X, y

X_train, y_train = normalize(train_df)
X_test,  y_test  = normalize(test_df)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
                          batch_size=256, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.tensor(X_test),  torch.tensor(y_test)),
                          batch_size=512)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

### SNN-related changes
#### Spike Encoding
Converting float samples to Bernoulli firing probability for spike readings [0,1] using `spikegen.rate`.
#### LIF neurons
Replacing ReLU used in the standard MLP with LIF neurons that maintain a membrane potential across timesteps and emit a binary spike when a threshold is crossed using `snn.Leaky`.
#### Timestep loop + spike-count readout
Instead of one forward pass, we loop over each timestep, feeding one "frame" of spikes through the network and accumulating output spikes. The class with the most output spikes wins.

In [ ]:
class AQI_SNN(nn.Module):
    def __init__(self, num_features=7, num_classes=6):
        super().__init__()
        self.fc1     = nn.Linear(num_features, 128)
        self.fc2     = nn.Linear(128, 64)
        self.fc_out  = nn.Linear(64, num_classes)

        # SNN Change 2
        self.lif1    = snn.Leaky(beta=0.95)
        self.lif2    = snn.Leaky(beta=0.90)
        self.lif_out = snn.Leaky(beta=0.85)

    # spikes: [NUM_STEPS, batch, 7]
    def forward(self, spikes):
        T = spikes.shape[0]
        mem1   = self.lif1.init_leaky()
        mem2   = self.lif2.init_leaky()
        mem_out = self.lif_out.init_leaky()
        out_rec = torch.zeros(spikes.shape[1], self.fc_out.out_features, device=spikes.device)

        # SNN Change 3
        for t in range(T):
            x = spikes[t]
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            cur_out = self.fc_out(spk2)
            spk_out, mem_out = self.lif_out(cur_out, mem_out)
            out_rec += spk_out

        return out_rec

In [ ]:
# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AQI_SNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # SNN Change 1
        spike_data = spikegen.rate(X_batch, num_steps=NUM_STEPS)

        out = model(spike_data)
        loss = criterion(out, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct += (out.argmax(1) == y_batch).sum().item()
        total += len(y_batch)

    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={total_loss/total:.4f}  acc={correct/total:.3f}")

In [ ]:
### Evaluation: spike-encoding + forward pass
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        spike_data = spikegen.rate(X_batch, num_steps=NUM_STEPS)
        out = model(spike_data)
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(y_batch.tolist())

present = sorted(set(all_labels) | set(all_preds))
target_names = [INT_TO_BUCKET[i] for i in present]

acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\nTest accuracy: {acc:.3f}")
print(f"Baseline (most-frequent class): {max(all_labels.count(c) for c in present) / len(all_labels):.3f}\n")
print(classification_report(all_labels, all_preds, labels=present, target_names=target_names))
print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds, labels=present))